# Coffee Standard J25 — visual annotation audit
Membuat contact sheet deterministik untuk 25 kelas pada seluruh candidate split, flag geometri, dan audit perbedaan label sibling. **Tidak melakukan training atau inference model.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import hashlib, importlib, json, os, shutil, subprocess, sys
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='codex/coffee-standard-primary-audit'
REMOTE='https://'+'github.com/ediprin/coffee-bean-detection.git'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1','--branch',BRANCH,REMOTE,str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO)],check=True)
sys.path.insert(0,str(REPO/'src')); importlib.invalidate_caches(); os.chdir(REPO)
from coffee_detector.analysis.coffee_standard_j25_visual_audit import audit_coffee_standard_j25_visuals
from coffee_detector.analysis.public_dataset_eligibility import extract_audit_archive
from coffee_detector.drive_project import resolve_drive_project_root
PROJECT=resolve_drive_project_root(required_relative_paths=('bundles/coffee-detection-with-standard-v8-yolov8.tar','datasets/coffee-standard-j25-grouped-v1/coffee_standard_j25_summary.json'))
ARCHIVE=PROJECT/'bundles/coffee-detection-with-standard-v8-yolov8.tar'
EXPECTED='5529de365ad888406b5534a5d1bf5a4a29c9937bc095b16174f8516d396bb1fc'
actual=hashlib.sha256(ARCHIVE.read_bytes()).hexdigest()
if actual!=EXPECTED: raise RuntimeError(f'SHA256 bundle salah: {actual}')
GROUPED=PROJECT/'datasets/coffee-standard-j25-grouped-v1'
RAW=extract_audit_archive(ARCHIVE,Path('/content/coffee-standard-v8-raw'))
OUTPUT=PROJECT/'evidence/coffee-standard-j25-visual-audit-v1'
print('REPO:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('INPUT VERIFIED:',actual)


In [ ]:
result=audit_coffee_standard_j25_visuals(GROUPED,RAW,OUTPUT,samples_per_class=3,flagged_limit=40)
print('DECISION:',result['decision'])
print('SELECTED CLASS OBJECTS:',result['selected_class_review_objects'])
print('GEOMETRY FLAGS:',result['geometry_flag_objects'])
print('SIBLING CONSISTENCY:',json.dumps(result['sibling_consistency'],indent=2,ensure_ascii=False))
print('TRAINING AUTHORIZED:',result['training_authorized'])
print('SUMMARY:',result['summary'])


In [ ]:
from IPython.display import Image as DisplayImage, display
for split,path in result['class_review_sheets'].items():
    print('CLASS REVIEW:',split,path)
    display(DisplayImage(filename=path,width=1400))
print('GEOMETRY FLAGS:',result['geometry_flag_sheet'])
display(DisplayImage(filename=result['geometry_flag_sheet'],width=1400))
print('Periksa box, konsistensi kelas lintas split, kelas hitam/coklat/sour, lubang, kulit/tanduk, ukuran, dan flag geometri.')
print('Kirim tiga class sheet, geometry sheet, dan SIBLING CONSISTENCY. Jangan training.')
